In [1]:
import json
import pandas as pd
import glob
import numpy as np

In [5]:
crowd = pd.read_csv('extracted_fb_data/all_h_scores.csv')
crowd.head(10)

,id,resolution_date,source,Type,resolved_to,resolved,SPF_forecast,SPF_bs,Pub_forecast,Pub_bs
0,45db5d06a001a6fa62eb9b23236adab43c56970d70a833...,2024-07-28,acled,Data,0.0,True,0.000,0.000000,0.05,0.0025
1,45db5d06a001a6fa62eb9b23236adab43c56970d70a833...,2024-08-20,acled,Data,0.0,True,0.000,0.000000,0.07,0.0049
2,45db5d06a001a6fa62eb9b23236adab43c56970d70a833...,2024-10-19,acled,Data,0.0,True,0.000,0.000000,0.08,0.0064
3,45db5d06a001a6fa62eb9b23236adab43c56970d70a833...,2025-01-17,acled,Data,0.0,True,0.000,0.000000,0.08,0.0064
4,45db5d06a001a6fa62eb9b23236adab43c56970d70a833...,2025-07-21,acled,Data,0.0,True,0.000,0.000000,0.09,0.0081
5,5713f8a61c04fa270a3a9e1791e4d9b5fa8e0d1cc1c1c2...,2024-07-28,acled,Data,0.0,True,0.055,0.003025,0.10,0.0100
6,5713f8a61c04fa270a3a9e1791e4d9b5fa8e0d1cc1c1c2...,2024-08-20,acled,Data,1.0,True,0.155,0.714025,0.10,0.8100
7,5713f8a61c04fa270a3a9e1791e4d9b5fa8e0d1cc1c1c2...,2024-10-19,acled,Data,1.0,True,0.160,0.705600,0.15,0.7225
8,5713f8a61c04fa270a3a9e1791e4d9b5fa8e0d1cc1c1c2...,2025-01-17,acled,Data,1.0,True,0.350,0.422500,0.18,0.6724
9,5713f8a61c04fa270a3a9e1791e4d9b5fa8e0d1cc1c1c2...,2025-07-21,acled,Data,0.0,True,0.160,0.025600,0.16,0.0256


In [6]:
def merge_llm_forecast(json_filepath, crowd_df, llm_name='llm_forecast'):
    with open(json_filepath, 'r', encoding='utf-8') as file:
        json_data = json.load(file)

    llm_df = pd.json_normalize(json_data, record_path=['forecasts'])
    llm_df = llm_df[~llm_df['id'].apply(lambda x: isinstance(x, list))]

    merged_df = pd.merge(
        crowd_df.copy(),
        llm_df[['id', 'resolution_date', 'forecast']],
        on=['id', 'resolution_date'],
        how='left',
        suffixes=('', f'_{llm_name}')
    )

    # Handle unmatched forecasts
    null_forecast_mask = merged_df['forecast'].isna()
    ids_to_match = merged_df.loc[null_forecast_mask, 'id'].unique()

    for id_val in ids_to_match:
        llm_forecasts = llm_df[llm_df['id'] == id_val]['forecast'].values

        if len(llm_forecasts) > 0:
            idx = merged_df[merged_df['id'] == id_val].index
            merged_df.loc[idx, 'forecast'] = llm_forecasts[0]

    merged_df = merged_df.rename(columns={'forecast': f'{llm_name}'})

    return merged_df

## OpenAI (GPTs)

### GPT4 Turbo 0409

**Baseline**: 2024-07-21.OpenAI.gpt_4_turbo_0409_zero_shot.json  
**reasoning**: 2024-07-21.OpenAI.gpt_4_turbo_0409_scratchpad.json  
**News**: 2024-07-21.OpenAI.gpt_4_turbo_0409_scratchpad_with_news.json  
**spf_p1**: 2024-07-21.OpenAI.gpt_4_turbo_0409_superforecaster_with_news_1.json  
**spf_p2**: 2024-07-21.OpenAI.gpt_4_turbo_0409_superforecaster_with_news_2.json  
**spf_p3**: 2024-07-21.OpenAI.gpt_4_turbo_0409_superforecaster_with_news_3.json

In [8]:
#0s
df1 = merge_llm_forecast(
    'ForecastBench-LLMs/GPT4-Turbo/2024-07-21.OpenAI.gpt_4_turbo_0409_zero_shot.json',
    crowd,
    llm_name='gpt4T_0S'
)
#Reas
df2 = merge_llm_forecast(
    'ForecastBench-LLMs/GPT4-Turbo/2024-07-21.OpenAI.gpt_4_turbo_0409_scratchpad.json',
    df1,
    llm_name='gpt4T_Reas'
)
#News
df3 = merge_llm_forecast(
    'ForecastBench-LLMs/GPT4-Turbo/2024-07-21.OpenAI.gpt_4_turbo_0409_scratchpad_with_news.json',
    df2,
    llm_name='gpt4T_News'
)
#spf1
df4 = merge_llm_forecast(
    'ForecastBench-LLMs/GPT4-Turbo/2024-07-21.OpenAI.gpt_4_turbo_0409_superforecaster_with_news_1.json',
    df3,
    llm_name='gpt4T_spf1'
)
#spf2
df5 = merge_llm_forecast(
    'ForecastBench-LLMs/GPT4-Turbo/2024-07-21.OpenAI.gpt_4_turbo_0409_superforecaster_with_news_2.json',
    df4,
    llm_name='gpt4T_spf2'
)
#spf3
df6 = merge_llm_forecast(
    'ForecastBench-LLMs/GPT4-Turbo/2024-07-21.OpenAI.gpt_4_turbo_0409_superforecaster_with_news_3.json',
    df5,
    llm_name='gpt4T_spf3'
)

In [9]:
df6['BS gpt4T_0S'] = (df6['gpt4T_0S'] - df6['resolved_to'])**2
df6['BS gpt4T_Reas'] = (df6['gpt4T_Reas'] - df6['resolved_to'])**2
df6['BS gpt4T_News'] = (df6['gpt4T_News'] - df6['resolved_to'])**2
df6['BS gpt4T_spf1'] = (df6['gpt4T_spf1'] - df6['resolved_to'])**2
df6['BS gpt4T_spf2'] = (df6['gpt4T_spf2'] - df6['resolved_to'])**2
df6['BS gpt4T_spf3'] = (df6['gpt4T_spf3'] - df6['resolved_to'])**2

#df6.to_excel('extracted_fb_data/GPT4Turbo.xlsx', index=False)
#df6.to_csv('extracted_fb_data/GPT4Turbo.csv', index=False)

### GPT4o

**Baseline**: 2024-07-21.OpenAI.gpt_4o_zero_shot.json  
**reasoning**: 2024-07-21.OpenAI.gpt_4o_scratchpad.json  
**News**: 2024-07-21.OpenAI.gpt_4o_scratchpad_with_news.json  
**spf_p1**: 2024-07-21.OpenAI.gpt_4o_superforecaster_with_news_1.json  
**spf_p2**: 2024-07-21.OpenAI.gpt_4o_superforecaster_with_news_2.json  
**spf_p3**: 2024-07-21.OpenAI.gpt_4o_superforecaster_with_news_3.json

In [10]:
#0s
df1 = merge_llm_forecast(
    'ForecastBench-LLMs/GPT4-o/2024-07-21.OpenAI.gpt_4o_zero_shot.json',
    crowd,
    llm_name='gpt4o_0S'
)
#Reas
df2 = merge_llm_forecast(
    'ForecastBench-LLMs/GPT4-o/2024-07-21.OpenAI.gpt_4o_scratchpad.json',
    df1,
    llm_name='gpt4o_Reas'
)
#News
df3 = merge_llm_forecast(
    'ForecastBench-LLMs/GPT4-o/2024-07-21.OpenAI.gpt_4o_scratchpad_with_news.json',
    df2,
    llm_name='gpt4o_News'
)
#spf1
df4 = merge_llm_forecast(
    'ForecastBench-LLMs/GPT4-o/2024-07-21.OpenAI.gpt_4o_superforecaster_with_news_1.json',
    df3,
    llm_name='gpt4o_spf1'
)
#spf2
df5 = merge_llm_forecast(
    'ForecastBench-LLMs/GPT4-o/2024-07-21.OpenAI.gpt_4o_superforecaster_with_news_2.json',
    df4,
    llm_name='gpt4o_spf2'
)
#spf3
df6 = merge_llm_forecast(
    'ForecastBench-LLMs/GPT4-o/2024-07-21.OpenAI.gpt_4o_superforecaster_with_news_3.json',
    df5,
    llm_name='gpt4o_spf3'
)

In [11]:
df6['BS gpt4o_0S'] = (df6['gpt4o_0S'] - df6['resolved_to'])**2
df6['BS gpt4o_Reas'] = (df6['gpt4o_Reas'] - df6['resolved_to'])**2
df6['BS gpt4o_News'] = (df6['gpt4o_News'] - df6['resolved_to'])**2
df6['BS gpt4o_spf1'] = (df6['gpt4o_spf1'] - df6['resolved_to'])**2
df6['BS gpt4o_spf2'] = (df6['gpt4o_spf2'] - df6['resolved_to'])**2
df6['BS gpt4o_spf3'] = (df6['gpt4o_spf3'] - df6['resolved_to'])**2

#df6.to_excel('extracted_fb_data/GPT4o.xlsx', index=False)
#df6.to_csv('extracted_fb_data/GPT4o.csv', index=False)

### GPT3.5

**Baseline**: 2024-07-21.OpenAI.gpt_3p5_turbo_0125_zero_shot.json  
**reasoning**: 2024-07-21.OpenAI.gpt_3p5_turbo_0125_scratchpad.json

In [13]:
#0s
df1 = merge_llm_forecast(
    'ForecastBench-LLMs/GPT3.5/2024-07-21.OpenAI.gpt_3p5_turbo_0125_zero_shot.json',
    crowd,
    llm_name='gpt3.5_0S'
)
#Reas
df2 = merge_llm_forecast(
    'ForecastBench-LLMs/GPT3.5/2024-07-21.OpenAI.gpt_3p5_turbo_0125_scratchpad.json',
    df1,
    llm_name='gpt3.5_Reas'
)

In [14]:
df2['BS gpt3.5_0S'] = (df2['gpt3.5_0S'] - df2['resolved_to'])**2
df2['BS gpt3.5_Reas'] = (df2['gpt3.5_Reas'] - df2['resolved_to'])**2

#df2.to_excel('extracted_fb_data/GPT3.5.xlsx', index=False)
#df2.to_csv('extracted_fb_data/GPT3.5.csv', index=False)

### GPT-4

**Baseline**: 2024-07-21.OpenAI.gpt_4_zero_shot.json  
**reasoning**: 2024-07-21.OpenAI.gpt_4_scratchpad.json

In [16]:
#0s
df1 = merge_llm_forecast(
    'ForecastBench-LLMs/GPT4/2024-07-21.OpenAI.gpt_4_zero_shot.json',
    crowd,
    llm_name='gpt4_0S'
)
#Reas
df2 = merge_llm_forecast(
    'ForecastBench-LLMs/GPT4/2024-07-21.OpenAI.gpt_4_scratchpad.json',
    df1,
    llm_name='gpt4_Reas'
)

In [17]:
df2['BS gpt4_0S'] = (df2['gpt4_0S'] - df2['resolved_to'])**2
df2['BS gpt4_Reas'] = (df2['gpt4_Reas'] - df2['resolved_to'])**2

#df2.to_excel('extracted_fb_data/GPT4.xlsx', index=False)
#df2.to_csv('extracted_fb_data/GPT4.csv', index=False)

## Anthropic (Claude)

### Claude 2.1

**Baseline**: 2024-07-21.Anthropic.claude_2p1_zero_shot.json  
**reasoning**: 2024-07-21.Anthropic.claude_2p1_scratchpad.json  
**News**: 2024-07-21.Anthropic.claude_2p1_scratchpad_with_news.json  
**spf_p1**: 2024-07-21.Anthropic.claude_2p1_superforecaster_with_news_1.json  
**spf_p2**: 2024-07-21.Anthropic.claude_2p1_superforecaster_with_news_2.json  
**spf_p3**: 2024-07-21.Anthropic.claude_2p1_superforecaster_with_news_3.json

In [18]:
#Claude 2.1 0s
df1 = merge_llm_forecast(
    'ForecastBench-LLMs/C2.1/2024-07-21.Anthropic.claude_2p1_zero_shot.json',
    crowd,
    llm_name='C2.1_0S'
)
#Claude 2.1 Reas
df2 = merge_llm_forecast(
    'ForecastBench-LLMs/C2.1/2024-07-21.Anthropic.claude_2p1_scratchpad.json',
    df1,
    llm_name='C2.1_Reas'
)
#Claude 2.1 News
df3 = merge_llm_forecast(
    'ForecastBench-LLMs/C2.1/2024-07-21.Anthropic.claude_2p1_scratchpad_with_news.json',
    df2,
    llm_name='C2.1_News'
)
#Claude 2.1 spf 1
df4 = merge_llm_forecast(
    'ForecastBench-LLMs/C2.1/2024-07-21.Anthropic.claude_2p1_superforecaster_with_news_1.json',
    df3,
    llm_name='C2.1_spf1'
)
#Claude 2.1 spf 2
df5 = merge_llm_forecast(
    'ForecastBench-LLMs/C2.1/2024-07-21.Anthropic.claude_2p1_superforecaster_with_news_2.json',
    df4,
    llm_name='C2.1_spf2'
)
#Claude 2.1 spf 3
df6 = merge_llm_forecast(
    'ForecastBench-LLMs/C2.1/2024-07-21.Anthropic.claude_2p1_superforecaster_with_news_3.json',
    df5,
    llm_name='C2.1_spf3'
)

In [19]:
df6['BS C2.1_0S'] = (df6['C2.1_0S'] - df6['resolved_to'])**2
df6['BS C2.1_Reas'] = (df6['C2.1_Reas'] - df6['resolved_to'])**2
df6['BS C2.1_News'] = (df6['C2.1_News'] - df6['resolved_to'])**2
df6['BS C2.1_spf1'] = (df6['C2.1_spf1'] - df6['resolved_to'])**2
df6['BS C2.1_spf2'] = (df6['C2.1_spf2'] - df6['resolved_to'])**2
df6['BS C2.1_spf3'] = (df6['C2.1_spf3'] - df6['resolved_to'])**2

#df6.to_excel('extracted_fb_data/Claude2.1.xlsx', index=False)
#df6.to_csv('extracted_fb_data/Claude2.1.csv', index=False)

### Claude 3 Opus

**Baseline**: 2024-07-21.Anthropic.claude_3_opus_zero_shot.json  
**reasoning**: 2024-07-21.Anthropic.claude_3_opus_scratchpad.json  
**News**: 2024-07-21.Anthropic.claude_3_opus_scratchpad_with_news.json  
**spf_p1**: 2024-07-21.Anthropic.claude_3_opus_superforecaster_with_news_1.json  
**spf_p2**: 2024-07-21.Anthropic.claude_3_opus_superforecaster_with_news_2.json  
**spf_p3**: 2024-07-21.Anthropic.claude_3_opus_superforecaster_with_news_3.json

In [20]:
#Claude 3 OPUS 0s
df1 = merge_llm_forecast(
    'ForecastBench-LLMs/C3-Opus/2024-07-21.Anthropic.claude_3_opus_zero_shot.json',
    crowd,
    llm_name='C3O_0S'
)
#Claude 3 OPUS Reas
df2 = merge_llm_forecast(
    'ForecastBench-LLMs/C3-Opus/2024-07-21.Anthropic.claude_3_opus_scratchpad.json',
    df1,
    llm_name='C3O_Reas'
)
#Claude 3 OPUS News
df3 = merge_llm_forecast(
    'ForecastBench-LLMs/C3-Opus/2024-07-21.Anthropic.claude_3_opus_scratchpad_with_news.json',
    df2,
    llm_name='C3O_News'
)
#Claude 3 OPUS spf1
df4 = merge_llm_forecast(
    'ForecastBench-LLMs/C3-Opus/2024-07-21.Anthropic.claude_3_opus_superforecaster_with_news_1.json',
    df3,
    llm_name='C3O_spf1'
)
#Claude 3 OPUS spf2
df5 = merge_llm_forecast(
    'ForecastBench-LLMs/C3-Opus/2024-07-21.Anthropic.claude_3_opus_superforecaster_with_news_2.json',
    df4,
    llm_name='C3O_spf2'
)
#Claude 3 OPUS spf3
df6 = merge_llm_forecast(
    'ForecastBench-LLMs/C3-Opus/2024-07-21.Anthropic.claude_3_opus_superforecaster_with_news_3.json',
    df5,
    llm_name='C3O_spf3'
)

In [21]:
df6['BS C3O_0S'] = (df6['C3O_0S'] - df6['resolved_to'])**2
df6['BS C3O_Reas'] = (df6['C3O_Reas'] - df6['resolved_to'])**2
df6['BS C3O_News'] = (df6['C3O_News'] - df6['resolved_to'])**2
df6['BS C3O_spf1'] = (df6['C3O_spf1'] - df6['resolved_to'])**2
df6['BS C3O_spf2'] = (df6['C3O_spf2'] - df6['resolved_to'])**2
df6['BS C3O_spf3'] = (df6['C3O_spf3'] - df6['resolved_to'])**2

#df6.to_excel('extracted_fb_data/Claude3Opus.xlsx', index=False)
#df6.to_csv('extracted_fb_data/Claude3Opus.csv', index=False)

### Claude 3 Haiku

**Baseline**: 2024-07-21.Anthropic.claude_3_haiku_zero_shot.json  
**reasoning**: 2024-07-21.Anthropic.claude_3_haiku_scratchpad.json  
**News**: 2024-07-21.Anthropic.claude_3_haiku_scratchpad_with_news.json  
**spf_p1**: 2024-07-21.Anthropic.claude_3_haiku_superforecaster_with_news_1.json  
**spf_p2**: 2024-07-21.Anthropic.claude_3_haiku_superforecaster_with_news_2.json  
**spf_p3**: 2024-07-21.Anthropic.claude_3_haiku_superforecaster_with_news_3.json

In [22]:
#Claude 3 HAIKU 0s
df1 = merge_llm_forecast(
    'ForecastBench-LLMs/C3-Haiku/2024-07-21.Anthropic.claude_3_haiku_zero_shot.json',
    crowd,
    llm_name='C3H_0S'
)
#Claude 3 HAIKU Reas
df2 = merge_llm_forecast(
    'ForecastBench-LLMs/C3-Haiku/2024-07-21.Anthropic.claude_3_haiku_scratchpad.json',
    df1,
    llm_name='C3H_Reas'
)
#Claude 3 HAIKU News
df3 = merge_llm_forecast(
    'ForecastBench-LLMs/C3-Haiku/2024-07-21.Anthropic.claude_3_haiku_scratchpad_with_news.json',
    df2,
    llm_name='C3H_News'
)
#Claude 3 HAIKU spf1
df4 = merge_llm_forecast(
    'ForecastBench-LLMs/C3-Haiku/2024-07-21.Anthropic.claude_3_haiku_superforecaster_with_news_1.json',
    df3,
    llm_name='C3H_spf1'
)
#Claude 3 HAIKU spf2
df5 = merge_llm_forecast(
    'ForecastBench-LLMs/C3-Haiku/2024-07-21.Anthropic.claude_3_haiku_superforecaster_with_news_2.json',
    df4,
    llm_name='C3H_spf2'
)
#Claude 3 HAIKU spf3
df6 = merge_llm_forecast(
    'ForecastBench-LLMs/C3-Haiku/2024-07-21.Anthropic.claude_3_haiku_superforecaster_with_news_3.json',
    df5,
    llm_name='C3H_spf3'
)

In [23]:
df6['BS C3H_0S'] = (df6['C3H_0S'] - df6['resolved_to'])**2
df6['BS C3H_Reas'] = (df6['C3H_Reas'] - df6['resolved_to'])**2
df6['BS C3H_News'] = (df6['C3H_News'] - df6['resolved_to'])**2
df6['BS C3H_spf1'] = (df6['C3H_spf1'] - df6['resolved_to'])**2
df6['BS C3H_spf2'] = (df6['C3H_spf2'] - df6['resolved_to'])**2
df6['BS C3H_spf3'] = (df6['C3H_spf3'] - df6['resolved_to'])**2

#df6.to_excel('extracted_fb_data/Claude3Haiku.xlsx', index=False)
#df6.to_csv('extracted_fb_data/Claude3Haiku.csv', index=False)

### Claude 3.5 Sonnet

**Baseline**: 2024-07-21.Anthropic.claude_3p5_sonnet_zero_shot.json  
**reasoning**: 2024-07-21.Anthropic.claude_3p5_sonnet_scratchpad.json  
**News**: 2024-07-21.Anthropic.claude_3p5_sonnet_scratchpad_with_news.json  
**spf_p1**: 2024-07-21.Anthropic.claude_3p5_sonnet_superforecaster_with_news_1.json  
**spf_p2**: 2024-07-21.Anthropic.claude_3p5_sonnet_superforecaster_with_news_2.json  
**spf_p3**: 2024-07-21.Anthropic.claude_3p5_sonnet_superforecaster_with_news_3.json

In [24]:
#Claude 3.5 0s
df1 = merge_llm_forecast(
    'ForecastBench-LLMs/C3.5-Sonnet/2024-07-21.Anthropic.claude_3p5_sonnet_zero_shot.json',
    crowd,
    llm_name='C3.5S_0S'
)
#Claude 3.5 Reas
df2 = merge_llm_forecast(
    'ForecastBench-LLMs/C3.5-Sonnet/2024-07-21.Anthropic.claude_3p5_sonnet_scratchpad.json',
    df1,
    llm_name='C3.5S_Reas'
)
#Claude 3.5 News
df3 = merge_llm_forecast(
    'ForecastBench-LLMs/C3.5-Sonnet/2024-07-21.Anthropic.claude_3p5_sonnet_scratchpad_with_news.json',
    df2,
    llm_name='C3.5S_News'
)
#Claude 3.5 spf1
df4 = merge_llm_forecast(
    'ForecastBench-LLMs/C3.5-Sonnet/2024-07-21.Anthropic.claude_3p5_sonnet_superforecaster_with_news_1.json',
    df3,
    llm_name='C3.5S_spf1'
)
#Claude 3.5 spf2
df5 = merge_llm_forecast(
    'ForecastBench-LLMs/C3.5-Sonnet/2024-07-21.Anthropic.claude_3p5_sonnet_superforecaster_with_news_2.json',
    df4,
    llm_name='C3.5S_spf2'
)
#Claude 3.5 spf3
df6 = merge_llm_forecast(
    'ForecastBench-LLMs/C3.5-Sonnet/2024-07-21.Anthropic.claude_3p5_sonnet_superforecaster_with_news_3.json',
    df5,
    llm_name='C3.5S_spf3'
)

In [25]:
df6['BS C3.5S_0S'] = (df6['C3.5S_0S'] - df6['resolved_to'])**2
df6['BS C3.5S_Reas'] = (df6['C3.5S_Reas'] - df6['resolved_to'])**2
df6['BS C3.5S_News'] = (df6['C3.5S_News'] - df6['resolved_to'])**2
df6['BS C3.5S_spf1'] = (df6['C3.5S_spf1'] - df6['resolved_to'])**2
df6['BS C3.5S_spf2'] = (df6['C3.5S_spf2'] - df6['resolved_to'])**2
df6['BS C3.5S_spf3'] = (df6['C3.5S_spf3'] - df6['resolved_to'])**2

#df6.to_excel('extracted_fb_data/Claude3.5Sonnet.xlsx', index=False)
#df6.to_csv('extracted_fb_data/Claude3.5Sonnet.csv', index=False)

## Google (Gemini)

### Gemini1.5 Flash

**Baseline**: 2024-07-21.Google.gemini_1p5_flash_zero_shot.json  
**reasoning**: 2024-07-21.Google.gemini_1p5_flash_scratchpad.json  
**News**: 2024-07-21.Google.gemini_1p5_flash_scratchpad_with_news.json  
**spf_p1**: 2024-07-21.Google.gemini_1p5_flash_superforecaster_with_news_1  
**spf_p2**: 2024-07-21.Google.gemini_1p5_flash_superforecaster_with_news_2  
**spf_p3**: 2024-07-21.Google.gemini_1p5_flash_superforecaster_with_news_3

In [26]:
#0s
df1 = merge_llm_forecast(
    'ForecastBench-LLMs/G1.5-Flash/2024-07-21.Google.gemini_1p5_flash_zero_shot.json',
    crowd,
    llm_name='G1.5f_0S'
)
#Reas
df2 = merge_llm_forecast(
    'ForecastBench-LLMs/G1.5-Flash/2024-07-21.Google.gemini_1p5_flash_scratchpad.json',
    df1,
    llm_name='G1.5f_Reas'
)
#News
df3 = merge_llm_forecast(
    'ForecastBench-LLMs/G1.5-Flash/2024-07-21.Google.gemini_1p5_flash_scratchpad_with_news.json',
    df2,
    llm_name='G1.5f_News'
)
#spf1
df4 = merge_llm_forecast(
    'ForecastBench-LLMs/G1.5-Flash/2024-07-21.Google.gemini_1p5_flash_superforecaster_with_news_1.json',
    df3,
    llm_name='G1.5f_spf1'
)
#spf2
df5 = merge_llm_forecast(
    'ForecastBench-LLMs/G1.5-Flash/2024-07-21.Google.gemini_1p5_flash_superforecaster_with_news_2.json',
    df4,
    llm_name='G1.5f_spf2'
)
#spf3
df6 = merge_llm_forecast(
    'ForecastBench-LLMs/G1.5-Flash/2024-07-21.Google.gemini_1p5_flash_superforecaster_with_news_3.json',
    df5,
    llm_name='G1.5f_spf3'
)

In [27]:
df6['BS G1.5f_0S'] = (df6['G1.5f_0S'] - df6['resolved_to'])**2
df6['BS G1.5f_Reas'] = (df6['G1.5f_Reas'] - df6['resolved_to'])**2
df6['BS G1.5f_News'] = (df6['G1.5f_News'] - df6['resolved_to'])**2
df6['BS G1.5f_spf1'] = (df6['G1.5f_spf1'] - df6['resolved_to'])**2
df6['BS G1.5f_spf2'] = (df6['G1.5f_spf2'] - df6['resolved_to'])**2
df6['BS G1.5f_spf3'] = (df6['G1.5f_spf3'] - df6['resolved_to'])**2

#df6.to_excel('extracted_fb_data/Gemini1.5Flash.xlsx', index=False)
#df6.to_csv('extracted_fb_data/Gemini1.5Flash.csv', index=False)

### Gemini1.5 Pro

**Baseline**: 2024-07-21.Google.gemini_1p5_pro_zero_shot.json  
**reasoning**: 2024-07-21.Google.gemini_1p5_pro_scratchpad.json  
**News**: 2024-07-21.Google.gemini_1p5_pro_scratchpad_with_news.json  
**spf_p1**:2024-07-21.Google.gemini_1p5_pro_superforecaster_with_news_1.json  
**spf_p2**: 2024-07-21.Google.gemini_1p5_pro_superforecaster_with_news_2.json  
**spf_p3**: 2024-07-21.Google.gemini_1p5_pro_superforecaster_with_news_3.json

In [28]:
#0s
df1 = merge_llm_forecast(
    'ForecastBench-LLMs/G1.5-Pro/2024-07-21.Google.gemini_1p5_pro_zero_shot.json',
    crowd,
    llm_name='G1.5P_0S'
)
#Reas
df2 = merge_llm_forecast(
    'ForecastBench-LLMs/G1.5-Pro/2024-07-21.Google.gemini_1p5_pro_scratchpad.json',
    df1,
    llm_name='G1.5P_Reas'
)
#News
df3 = merge_llm_forecast(
    'ForecastBench-LLMs/G1.5-Pro/2024-07-21.Google.gemini_1p5_pro_scratchpad_with_news.json',
    df2,
    llm_name='G1.5P_News'
)
#spf1
df4 = merge_llm_forecast(
    'ForecastBench-LLMs/G1.5-Pro/2024-07-21.Google.gemini_1p5_pro_superforecaster_with_news_1.json',
    df3,
    llm_name='G1.5P_spf1'
)
#spf2
df5 = merge_llm_forecast(
    'ForecastBench-LLMs/G1.5-Pro/2024-07-21.Google.gemini_1p5_pro_superforecaster_with_news_2.json',
    df4,
    llm_name='G1.5P_spf2'
)
#spf3
df6 = merge_llm_forecast(
    'ForecastBench-LLMs/G1.5-Pro/2024-07-21.Google.gemini_1p5_pro_superforecaster_with_news_3.json',
    df5,
    llm_name='G1.5P_spf3'
)

In [29]:
df6['BS G1.5P_0S'] = (df6['G1.5P_0S'] - df6['resolved_to'])**2
df6['BS G1.5P_Reas'] = (df6['G1.5P_Reas'] - df6['resolved_to'])**2
df6['BS G1.5P_News'] = (df6['G1.5P_News'] - df6['resolved_to'])**2
df6['BS G1.5P_spf1'] = (df6['G1.5P_spf1'] - df6['resolved_to'])**2
df6['BS G1.5P_spf2'] = (df6['G1.5P_spf2'] - df6['resolved_to'])**2
df6['BS G1.5P_spf3'] = (df6['G1.5P_spf3'] - df6['resolved_to'])**2

#df6.to_excel('extracted_fb_data/Gemini1.5Pro.xlsx', index=False)
#df6.to_csv('extracted_fb_data/Gemini1.5Pro.csv', index=False)

## Meta (Llama)

### llama 2 70B

**Baseline**: 2024-07-21.Meta.llama_2_70b_zero_shot.json  
**reasoning**: 2024-07-21.Meta.llama_2_70b_scratchpad.json

In [30]:
#0s
df1 = merge_llm_forecast(
    'ForecastBench-LLMs/L2.70b/2024-07-21.Meta.llama_2_70b_zero_shot.json',
    crowd,
    llm_name='L2.70b_0S'
)
#Reas
df2 = merge_llm_forecast(
    'ForecastBench-LLMs/l2.70b/2024-07-21.Meta.llama_2_70b_scratchpad.json',
    df1,
    llm_name='L2.70b_Reas'
)

In [31]:
df2['BS L2.70b_0S'] = (df2['L2.70b_0S'] - df2['resolved_to'])**2
df2['BS L2.70b_Reas'] = (df2['L2.70b_Reas'] - df2['resolved_to'])**2

#df2.to_excel('extracted_fb_data/Llama2.70B.xlsx', index=False)
#df2.to_csv('extracted_fb_data/Llama2.70B.csv', index=False)

### llama 3 8B

**Baseline**: 2024-07-21.Meta.llama_3_8b_zero_shot.json  
**reasoning**: 2024-07-21.Meta.llama_3_8b_scratchpad.json

In [32]:
#0s
df1 = merge_llm_forecast(
    'ForecastBench-LLMs/L3.8b/2024-07-21.Meta.llama_3_8b_zero_shot.json',
    crowd,
    llm_name='L3.8b_0S'
)
#Reas
df2 = merge_llm_forecast(
    'ForecastBench-LLMs/L3.8b/2024-07-21.Meta.llama_3_8b_scratchpad.json',
    df1,
    llm_name='L3.8b_Reas'
)

In [33]:
df2['BS L3.8b_0S'] = (df2['L3.8b_0S'] - df2['resolved_to'])**2
df2['BS L3.8b_Reas'] = (df2['L3.8b_Reas'] - df2['resolved_to'])**2

#df2.to_excel('extracted_fb_data/Llama3.8B.xlsx', index=False)
#df2.to_csv('extracted_fb_data/Llama3.8B.csv', index=False)

### llama 3 70B

**Baseline**: 2024-07-21.Meta.llama_3_70b_zero_shot.json  
**reasoning**: 2024-07-21.Meta.llama_3_70b_scratchpad.json

In [34]:
#0s
df1 = merge_llm_forecast(
    'ForecastBench-LLMs/L3.70b/2024-07-21.Meta.llama_3_70b_zero_shot.json',
    crowd,
    llm_name='L3.70b_0S'
)
#Reas
df2 = merge_llm_forecast(
    'ForecastBench-LLMs/L3.70b/2024-07-21.Meta.llama_3_70b_scratchpad.json',
    df1,
    llm_name='L3.70b_Reas'
)

In [35]:
df2['BS L3.70b_0S'] = (df2['L3.70b_0S'] - df2['resolved_to'])**2
df2['BS L3.70b_Reas'] = (df2['L3.70b_Reas'] - df2['resolved_to'])**2

#df2.to_excel('extracted_fb_data/Llama3.70B.xlsx', index=False)
#df2.to_csv('extracted_fb_data/Llama3.70B.csv', index=False)

## Mistral

### Mistral Large

**Baseline**: 2024-07-21.Mistral AI.mistral_large_zero_shot.json  
**reasoning**: 2024-07-21.Mistral AI.mistral_large_scratchpad.json  
**News**: 2024-07-21.Mistral AI.mistral_large_scratchpad_with_news.json  
**spf_p1**: 2024-07-21.Mistral AI.mistral_large_superforecaster_with_news_1.json  
**spf_p2**: 2024-07-21.Mistral AI.mistral_large_superforecaster_with_news_2.json  
**spf_p3**: 2024-07-21.Mistral AI.mistral_large_superforecaster_with_news_3.json

In [36]:
#0s
df1 = merge_llm_forecast(
    'ForecastBench-LLMs/M-Large/2024-07-21.Mistral AI.mistral_large_zero_shot.json',
    crowd,
    llm_name='ML_0S'
)
#Reas
df2 = merge_llm_forecast(
    'ForecastBench-LLMs/M-Large/2024-07-21.Mistral AI.mistral_large_scratchpad.json',
    df1,
    llm_name='ML_Reas'
)
#News
df3 = merge_llm_forecast(
    'ForecastBench-LLMs/M-Large/2024-07-21.Mistral AI.mistral_large_scratchpad_with_news.json',
    df2,
    llm_name='ML_News'
)
#spf1
df4 = merge_llm_forecast(
    'ForecastBench-LLMs/M-Large/2024-07-21.Mistral AI.mistral_large_superforecaster_with_news_1.json',
    df3,
    llm_name='ML_spf1'
)
#spf2
df5 = merge_llm_forecast(
    'ForecastBench-LLMs/M-Large/2024-07-21.Mistral AI.mistral_large_superforecaster_with_news_2.json',
    df4,
    llm_name='ML_spf2'
)
#spf3
df6 = merge_llm_forecast(
    'ForecastBench-LLMs/M-Large/2024-07-21.Mistral AI.mistral_large_superforecaster_with_news_3.json',
    df5,
    llm_name='ML_spf3'
)

In [37]:
df6['BS ML_0S'] = (df6['ML_0S'] - df6['resolved_to'])**2
df6['BS ML_Reas'] = (df6['ML_Reas'] - df6['resolved_to'])**2
df6['BS ML_News'] = (df6['ML_News'] - df6['resolved_to'])**2
df6['BS ML_spf1'] = (df6['ML_spf1'] - df6['resolved_to'])**2
df6['BS ML_spf2'] = (df6['ML_spf2'] - df6['resolved_to'])**2
df6['BS ML_spf3'] = (df6['ML_spf3'] - df6['resolved_to'])**2

#df6.to_excel('extracted_fb_data/MistralLarge.xlsx', index=False)
#df6.to_csv('extracted_fb_data/MistralLarge.csv', index=False)

### Mistral 8x22B

**Baseline**: 2024-07-21.Mistral AI.mistral_8x22b_instruct_zero_shot.json  
**reasoning**: 2024-07-21.Mistral AI.mistral_8x22b_instruct_scratchpad.json  
**News**: 2024-07-21.Mistral AI.mistral_8x22b_instruct_scratchpad_with_news.json  
**spf_p1**: 2024-07-21.Mistral AI.mistral_8x22b_instruct_superforecaster_with_news_1.json  
**spf_p2**: 2024-07-21.Mistral AI.mistral_8x22b_instruct_superforecaster_with_news_2.json  
**spf_p3**: 2024-07-21.Mistral AI.mistral_8x22b_instruct_superforecaster_with_news_3.json

In [38]:
#0s
df1 = merge_llm_forecast(
    'ForecastBench-LLMs/M8x22b/2024-07-21.Mistral AI.mistral_8x22b_instruct_zero_shot.json',
    crowd,
    llm_name='M22B_0S'
)
#Reas
df2 = merge_llm_forecast(
    'ForecastBench-LLMs/M8x22b/2024-07-21.Mistral AI.mistral_8x22b_instruct_scratchpad.json',
    df1,
    llm_name='M22B_Reas'
)
#News
df3 = merge_llm_forecast(
    'ForecastBench-LLMs/M8x22b/2024-07-21.Mistral AI.mistral_8x22b_instruct_scratchpad_with_news.json',
    df2,
    llm_name='M22B_News'
)
#spf1
df4 = merge_llm_forecast(
    'ForecastBench-LLMs/M8x22b/2024-07-21.Mistral AI.mistral_8x22b_instruct_superforecaster_with_news_1.json',
    df3,
    llm_name='M22B_spf1'
)
#spf2
df5 = merge_llm_forecast(
    'ForecastBench-LLMs/M8x22b/2024-07-21.Mistral AI.mistral_8x22b_instruct_superforecaster_with_news_2.json',
    df4,
    llm_name='M22B_spf2'
)
#spf3
df6 = merge_llm_forecast(
    'ForecastBench-LLMs/M8x22b/2024-07-21.Mistral AI.mistral_8x22b_instruct_superforecaster_with_news_3.json',
    df5,
    llm_name='M22B_spf3'
)

In [39]:
df6['BS M22B_0S'] = (df6['M22B_0S'] - df6['resolved_to'])**2
df6['BS M22B_Reas'] = (df6['M22B_Reas'] - df6['resolved_to'])**2
df6['BS M22B_News'] = (df6['M22B_News'] - df6['resolved_to'])**2
df6['BS M22B_spf1'] = (df6['M22B_spf1'] - df6['resolved_to'])**2
df6['BS M22B_spf2'] = (df6['M22B_spf2'] - df6['resolved_to'])**2
df6['BS M22B_spf3'] = (df6['M22B_spf3'] - df6['resolved_to'])**2

#df6.to_excel('extracted_fb_data/Mistral8x22B.xlsx', index=False)
#df6.to_csv('extracted_fb_data/Mistral8x22B.csv', index=False)

### Mistral 8x7b

**Baseline**: 2024-07-21.Mistral AI.mistral_8x7b_instruct_zero_shot.json  
**reasoning**: 2024-07-21.Mistral AI.mistral_8x7b_instruct_scratchpad.json  
**News**: 2024-07-21.Mistral AI.mistral_8x7b_instruct_scratchpad_with_news.json  
**spf_p1**: 2024-07-21.Mistral AI.mistral_8x7b_instruct_superforecaster_with_news_1.json  
**spf_p2**: 2024-07-21.Mistral AI.mistral_8x7b_instruct_superforecaster_with_news_2.json  
**spf_p3**: 2024-07-21.Mistral AI.mistral_8x7b_instruct_superforecaster_with_news_3.json

In [40]:
#0s
df1 = merge_llm_forecast(
    'ForecastBench-LLMs/M8x7b/2024-07-21.Mistral AI.mistral_8x7b_instruct_zero_shot.json',
    crowd,
    llm_name='M7B_0S'
)
#Reas
df2 = merge_llm_forecast(
    'ForecastBench-LLMs/M8x7b/2024-07-21.Mistral AI.mistral_8x7b_instruct_scratchpad.json',
    df1,
    llm_name='M7B_Reas'
)
#News
df3 = merge_llm_forecast(
    'ForecastBench-LLMs/M8x7b/2024-07-21.Mistral AI.mistral_8x7b_instruct_scratchpad_with_news.json',
    df2,
    llm_name='M7B_News'
)
#spf1
df4 = merge_llm_forecast(
    'ForecastBench-LLMs/M8x7b/2024-07-21.Mistral AI.mistral_8x7b_instruct_superforecaster_with_news_1.json',
    df3,
    llm_name='M7B_spf1'
)
#spf2
df5 = merge_llm_forecast(
    'ForecastBench-LLMs/M8x7b/2024-07-21.Mistral AI.mistral_8x7b_instruct_superforecaster_with_news_2.json',
    df4,
    llm_name='M7B_spf2'
)
#spf3
df6 = merge_llm_forecast(
    'ForecastBench-LLMs/M8x7b/2024-07-21.Mistral AI.mistral_8x7b_instruct_superforecaster_with_news_3.json',
    df5,
    llm_name='M7B_spf3'
)

In [41]:
df6['BS M7B_0S'] = (df6['M7B_0S'] - df6['resolved_to'])**2
df6['BS M7B_Reas'] = (df6['M7B_Reas'] - df6['resolved_to'])**2
df6['BS M7B_News'] = (df6['M7B_News'] - df6['resolved_to'])**2
df6['BS M7B_spf1'] = (df6['M7B_spf1'] - df6['resolved_to'])**2
df6['BS M7B_spf2'] = (df6['M7B_spf2'] - df6['resolved_to'])**2
df6['BS M7B_spf3'] = (df6['M7B_spf3'] - df6['resolved_to'])**2

#df6.to_excel('extracted_fb_data/Mistral8x7B.xlsx', index=False)
#df6.to_csv('extracted_fb_data/Mistral8x7B.csv', index=False)

# ALL IN ONE (ACE DATA)

In [42]:
# 16 LLMs
gpt4t = pd.read_csv('extracted_fb_data/GPT4Turbo.csv')
gpt4o = pd.read_csv('extracted_fb_data/GPT4o.csv')
gpt3_5 = pd.read_csv('extracted_fb_data/GPT3.5.csv')
gpt4 = pd.read_csv('extracted_fb_data/GPT4.csv')
c_2_1 = pd.read_csv('extracted_fb_data/Claude2.1.csv')
c_3H = pd.read_csv('extracted_fb_data/Claude3Haiku.csv')
c_3O = pd.read_csv('extracted_fb_data/Claude3Opus.csv')
c_3_5 = pd.read_csv('extracted_fb_data/Claude3.5Sonnet.csv')
gemFlash= pd.read_csv('extracted_fb_data/Gemini1.5Flash.csv')
gemPro = pd.read_csv('extracted_fb_data/Gemini1.5Pro.csv')
llama2_70 = pd.read_csv('extracted_fb_data/Llama2.70B.csv')
llama3_8 = pd.read_csv('extracted_fb_data/Llama3.8B.csv')
llama3_70 = pd.read_csv('extracted_fb_data/Llama3.70B.csv')
ml = pd.read_csv('extracted_fb_data/MistralLarge.csv')
m22b = pd.read_csv('extracted_fb_data/Mistral8x22B.csv')
m7b = pd.read_csv('extracted_fb_data/Mistral8x7B.csv')

In [43]:
llms = [gpt4t, gpt4o, gpt3_5, gpt4, c_2_1, c_3H, c_3O, c_3_5, gemFlash, gemPro, llama2_70, llama3_8, llama3_70, ml, m22b, m7b]

In [45]:
# human forecasts AND human Brier Scores

all_in_one_bs = crowd[['id', 'resolution_date', 'source', 'Type', 'resolved_to', 
                   'SPF_forecast', 'Pub_forecast', 'SPF_bs', 'Pub_bs']].copy()

# columns to IGNORE (Metadata + Human data we already have)
cols_to_exclude = [
    'source', 'Type', 'resolved_to', 'resolved', 
    'SPF_forecast', 'SPF_bs', 'Pub_forecast', 'Pub_bs'
]


for df in llms:
    cols_to_keep = [
        col for col in df.columns 
        if col not in cols_to_exclude 
        and col not in ['id', 'resolution_date']
    ]
    
    subset = df[['id', 'resolution_date'] + cols_to_keep]
    
    # merge
    all_in_one_bs = pd.merge(
        all_in_one_bs, 
        subset, 
        on=['id', 'resolution_date'], 
        how='left'
    )

print(f"Shape: {all_in_one_bs.shape}")
print(all_in_one_bs.columns.tolist())

Shape: (580, 161)
['id', 'resolution_date', 'source', 'Type', 'resolved_to', 'SPF_forecast', 'Pub_forecast', 'SPF_bs', 'Pub_bs', 'gpt4T_0S', 'gpt4T_Reas', 'gpt4T_News', 'gpt4T_spf1', 'gpt4T_spf2', 'gpt4T_spf3', 'BS gpt4T_0S', 'BS gpt4T_Reas', 'BS gpt4T_News', 'BS gpt4T_spf1', 'BS gpt4T_spf2', 'BS gpt4T_spf3', 'gpt4o_0S', 'gpt4o_Reas', 'gpt4o_News', 'gpt4o_spf1', 'gpt4o_spf2', 'gpt4o_spf3', 'BS gpt4o_0S', 'BS gpt4o_Reas', 'BS gpt4o_News', 'BS gpt4o_spf1', 'BS gpt4o_spf2', 'BS gpt4o_spf3', 'gpt3.5_0S', 'gpt3.5_Reas', 'BS gpt3.5_0S', 'BS gpt3.5_Reas', 'gpt4_0S', 'gpt4_Reas', 'BS gpt4_0S', 'BS gpt4_Reas', 'C2.1_0S', 'C2.1_Reas', 'C2.1_News', 'C2.1_spf1', 'C2.1_spf2', 'C2.1_spf3', 'BS C2.1_0S', 'BS C2.1_Reas', 'BS C2.1_News', 'BS C2.1_spf1', 'BS C2.1_spf2', 'BS C2.1_spf3', 'C3H_0S', 'C3H_Reas', 'C3H_News', 'C3H_spf1', 'C3H_spf2', 'C3H_spf3', 'BS C3H_0S', 'BS C3H_Reas', 'BS C3H_News', 'BS C3H_spf1', 'BS C3H_spf2', 'BS C3H_spf3', 'C3O_0S', 'C3O_Reas', 'C3O_News', 'C3O_spf1', 'C3O_spf2', 'C3O_

In [46]:
#all_in_one_bs.to_csv('all_h_m.csv', index=False)